In [1]:
# notebooks/corpus_stat
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

In [2]:
sns.set_theme(style="whitegrid", palette="Blues_d")
Path("results/figures").mkdir(parents=True, exist_ok=True)

df = pd.read_csv("/Users/muhammadmomotazulislam/Documents/sib-batterybert/data/metadata/sib_paper_database.csv")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df[df["year"].between(2015, 2025)]

print("=" * 55)
print("SIB-BatteryBERT Corpus Statistics")
print("=" * 55)
print(f"Total unique papers:        {len(df):>10,}")
print(f"With abstracts:             {df.abstract.notna().sum():>10,}")
print(f"Open access:                {df.is_oa.sum():>10,}")
print(f"With PDF URL:               {df.pdf_url.notna().sum():>10,}")
#print(f"Unique journals:            {df.journal.nunique():>10,}")
print(f"Year range:                 {int(df.year.min()):>10} - {int(df.year.max())}")
print("=" * 55)

SIB-BatteryBERT Corpus Statistics
Total unique papers:            16,446
With abstracts:                  8,482
Open access:                     3,439
With PDF URL:                    3,439
Year range:                       2015 - 2025


In [3]:
# ── CHART 1: Papers per year ──────────────────────────
yearly = df.groupby("year").size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(yearly["year"], yearly["count"], color="#2E86C1", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Publication Year", fontsize=12)
ax.set_ylabel("Number of Papers", fontsize=12)
ax.set_title("Sodium-Ion Battery Research: Exponential Growth (2015–2025)", fontsize=13, fontweight="bold")
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

# Annotate the most recent bar
last = yearly.iloc[-1]
ax.annotate(f"{int(last['count']):,} papers", xy=(last["year"], last["count"]),
            xytext=(0, 8), textcoords="offset points", ha="center", fontsize=10, color="#1B4F72")

plt.tight_layout()
plt.savefig("results/figures/papers_per_year.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results/figures/papers_per_year.png")

Saved: results/figures/papers_per_year.png


In [4]:
# ── CHART 2: Papers by Source ──────────────────────────
source_counts = df["source"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
source_counts.plot(kind="barh", ax=ax, color="#2E86C1")
ax.set_xlabel("Number of Papers", fontsize=12)
ax.set_title("Papers by Data Source in SIB Corpus", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("results/figures/papers_by_source.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results/figures/papers_by_source.png")

Saved: results/figures/papers_by_source.png


In [9]:
# ── CHART 3: Open-access ratio by year ────────────────
# Step 1: fix is_oa dtype on the main df FIRST
df["is_oa"] = df["is_oa"].map({"True": True, "False": False, True: True, False: False})
df["is_oa"] = pd.to_numeric(df["is_oa"], errors="coerce").fillna(0).astype(int)

# Step 2: NOW do the groupby
oa_by_year = df.groupby("year").agg(
    total=("doi", "count"),
    oa=("is_oa", "sum")
).reset_index()
oa_by_year["oa_pct"] = oa_by_year["oa"] / oa_by_year["total"] * 100

# Step 3: clean year column
oa_by_year = oa_by_year.dropna(subset=["year"])
oa_by_year["year"] = pd.to_numeric(oa_by_year["year"], errors="coerce").astype(int)
oa_by_year = oa_by_year[oa_by_year["year"].between(2015, 2025)]
oa_by_year["oa_pct"] = pd.to_numeric(oa_by_year["oa_pct"], errors="coerce")
oa_by_year = oa_by_year.dropna(subset=["year", "oa_pct"])

# Step 4: plot
x = oa_by_year["year"].to_numpy(dtype=float)
y = oa_by_year["oa_pct"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(x, y, alpha=0.3, color="#1E8449")
ax.plot(x, y, color="#1E8449", linewidth=2.5)
ax.set_ylim(0, 100)
ax.set_ylabel("Open Access %", fontsize=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_title("Open Access Rate Over Time — Corpus is Legally Clean",
             fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("results/figures/oa_ratio_by_year.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results/figures/oa_ratio_by_year.png")

Saved: results/figures/oa_ratio_by_year.png
